# CM3065 — Exercise 4 (Vosk Offline ASR)

This notebook uses **Vosk (offline ASR)** with **language selection (EN/IT/ES)**, basic **noise-robust preprocessing**, and **WER evaluation**.


## Paths (robust)

- Set `BASE_DIR` if needed.
- The notebook will try to **auto-detect** the correct `AUDIO_DIR` if the expected WAVs are in a nearby folder.


In [3]:
from pathlib import Path

# Imports Path to handle filesystem locations using OS-safe path objects instead of raw strings.

# Defines the root folder that contains the language subfolders (EN / ES / IT) and their audio/model contents.
# Path() stores the location as an object so joins and checks work consistently across operations.
AUDIO_DIR = Path("/Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files")

# Builds a dictionary that maps a short language key to the expected Vosk model directory for that language.
# Each model path is constructed by joining:
# - the audio root
# - the language folder (EN / ES / IT)
# - the specific model folder name for that language
VOSK_MODEL_PATHS = {
    "en": AUDIO_DIR / "EN" / "model-en",
    "es": AUDIO_DIR / "ES" / "model-es",
    "it": AUDIO_DIR / "IT" / "model-it",
}

# Prints the resolved audio root so the active working directory can be confirmed at runtime.
print("AUDIO_DIR:", AUDIO_DIR)

# Counts how many .wav files exist in each language folder.
# glob("*.wav") returns an iterator of matching paths; list(...) materializes it so len(...) can count items.
print("EN wavs:", len(list((AUDIO_DIR / "EN").glob("*.wav"))))
print("ES wavs:", len(list((AUDIO_DIR / "ES").glob("*.wav"))))
print("IT wavs:", len(list((AUDIO_DIR / "IT").glob("*.wav"))))

# Prints a header label before listing model validation results.
print("\nModel paths:")

# Iterates through each language key and its model path to verify required model structure on disk.
for k, v in VOSK_MODEL_PATHS.items():
    # For each model folder:
    # - v.exists() checks the model directory exists
    # - (v/"am").exists() checks the acoustic model subfolder is present
    # - (v/"conf").exists() checks configuration files folder is present
    # These checks confirm the model is likely complete and loadable by Vosk.
    print(k, "->", v, "| exists:", v.exists(), "| am:", (v/"am").exists(), "| conf:", (v/"conf").exists())


AUDIO_DIR: /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files
EN wavs: 7
ES wavs: 5
IT wavs: 5

Model paths:
en -> /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/EN/model-en | exists: True | am: True | conf: True
es -> /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/ES/model-es | exists: True | am: True | conf: True
it -> /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/IT/model-it | exists: True | am: True | conf: True


In [4]:
%pip install -U vosk


Note: you may need to restart the kernel to use updated packages.


In [5]:
##importing the libraries
from vosk import Model, KaldiRecognizer
import json

import numpy as np
import math
import re

from scipy.io import wavfile
from scipy import signal
import pandas as pd
from pathlib import Path


In [6]:
# Maps each audio filename to its expected ground-truth transcript.
# These reference sentences act as the target text for evaluation, allowing recognition output to be compared.
REFERENCE = {
    # English reference transcripts for the EN audio set.
    'checkin.wav': 'Where is the check-in desk?',
    'parents.wav': 'I have lost my parents.',
    'suitcase.wav': 'Please, I have lost my suitcase.',
    'what_time.wav': 'What time is my plane?',
    'where.wav': 'Where are the restaurants and shops?',
   

    # Italian reference transcripts for the IT audio set.
    'checkin_it.wav': "Dove e' il bancone?",
    'parents_it.wav': 'Ho perso i miei genitori.',
    'suitcase_it.wav': 'Per favore, ho perso la mia valigia.',
    'what_time_it.wav': "A che ora e’ il mio aereo?",
    'where_it.wav': 'Dove sono i ristoranti e i negozi?',

    # Spanish reference transcripts for the ES audio set.
    'checkin_es.wav': '¿Dónde están los mostradores?',
    'parents_es.wav': 'He perdido a mis padres.',
    'suitcase_es.wav': 'Por favor, he perdido mi maleta.',
    'what_time_es.wav': '¿A qué hora es mi avión?',
    'where_es.wav': '¿Dónde están los restaurantes y las tiendas?',
}

# Maps each audio filename to the language code that should be used for recognition.
# This lookup ensures the correct Vosk model is selected for each file during batch processing.
LANG_BY_FILE = {
    # English files should be processed with the English model.
    'checkin.wav': 'en',
    'parents.wav': 'en',
    'suitcase.wav': 'en',
    'what_time.wav': 'en',
    'where.wav': 'en',

    # Italian files should be processed with the Italian model.
    'checkin_it.wav': 'it',
    'parents_it.wav': 'it',
    'suitcase_it.wav': 'it',
    'what_time_it.wav': 'it',
    'where_it.wav': 'it',

    # Spanish files should be processed with the Spanish model.
    'checkin_es.wav': 'es',
    'parents_es.wav': 'es',
    'suitcase_es.wav': 'es',
    'what_time_es.wav': 'es',
    'where_es.wav': 'es',
}


In [7]:
# Sets the target sample rate expected by the speech recognition pipeline.
# 16 kHz is a common Vosk-friendly rate and reduces unnecessary bandwidth while preserving speech clarity.
TARGET_SR = 16000

# Enables or disables an optional speech-focused bandpass filter stage.
# When enabled, frequency content outside typical speech range is attenuated before recognition.
USE_BANDPASS = True

# Converts an audio array into a single mono channel.
# If the input is already 1D, it is treated as mono and returned unchanged.
# If the input has multiple channels, channels are averaged to produce a single waveform.
def to_mono(x: np.ndarray) -> np.ndarray:
    if x.ndim == 1:
        return x
    return x.mean(axis=1)

# Resamples audio from sr_in to sr_out using polyphase filtering.
# The up/down ratios are reduced using the greatest common divisor to keep the resampling ratio minimal.
# resample_poly applies an efficient anti-aliasing filter suitable for speech preprocessing.
def resample_audio(x: np.ndarray, sr_in: int, sr_out: int) -> np.ndarray:
    if sr_in == sr_out:
        return x
    g = math.gcd(sr_in, sr_out)
    up = sr_out // g
    down = sr_in // g
    return signal.resample_poly(x, up, down)

# Normalizes audio loudness by matching its RMS energy to a target value.
# This stabilizes recognition performance across files recorded at different volumes.
# A small epsilon is included to avoid divide-by-zero when signals contain silence.
def rms_normalize(x: np.ndarray, target_rms: float = 0.03) -> np.ndarray:
    x = x.astype(np.float32)
    rms = np.sqrt(np.mean(x**2) + 1e-12)
    if rms <= 0:
        return x
    gain = target_rms / rms
    x = x * gain
    return np.clip(x, -1.0, 1.0)

# Applies a bandpass filter tuned to the typical speech frequency range.
# Low cut removes rumble and handling noise; high cut removes high-frequency hiss and non-speech content.
# Butterworth design provides a smooth response suitable for speech cleanup.
def bandpass_speech(x: np.ndarray, sr: int, low_hz: float = 300.0, high_hz: float = 3400.0) -> np.ndarray:
    nyq = 0.5 * sr
    low = max(low_hz / nyq, 1e-5)
    high = min(high_hz / nyq, 0.999)
    b, a = signal.butter(4, [low, high], btype='bandpass')
    return signal.lfilter(b, a, x)

# Loads a WAV file, converts it into a clean, standardized int16 waveform suitable for recognition.
# Processing steps include:
# - decoding PCM into float range
# - mono conversion
# - resampling to TARGET_SR
# - RMS normalization
# - optional bandpass filtering + re-normalization
# Output is returned as int16 PCM to match common speech-recognition input expectations.
def load_and_prepare_wav_i16(path: Path, target_sr: int = TARGET_SR) -> np.ndarray:
    sr, x = wavfile.read(str(path))

    # Converts common PCM integer encodings into normalized float samples in approximately [-1, 1].
    # int16 and int32 are scaled by their maximum representable magnitudes.
    if x.dtype == np.int16:
        x = x.astype(np.float32) / 32768.0
    elif x.dtype == np.int32:
        x = x.astype(np.float32) / 2147483648.0
    else:
        x = x.astype(np.float32)

    # Ensures a single-channel signal so later processing and recognition are consistent.
    x = to_mono(x)

    # Resamples to the configured target rate so all files share the same timebase and model expectations.
    x = resample_audio(x, sr, target_sr)

    # Normalizes loudness so recognition is less sensitive to recording volume differences.
    x = rms_normalize(x)

    # Optionally filters to speech band to reduce noise and improve intelligibility for the recognizer.
    if USE_BANDPASS:
        x = bandpass_speech(x, target_sr)

        # Re-normalizes after filtering because band-limiting changes overall signal energy.
        x = rms_normalize(x)

    # Converts the processed float signal back into int16 PCM for downstream speech recognition input.
    return (x * 32767.0).astype(np.int16)


In [8]:
# Stores already-loaded Vosk models in memory so repeated transcriptions do not reload models from disk.
# This reduces startup overhead when processing many files across multiple languages.
MODEL_CACHE = {}

# Loads a Vosk speech recognition model for a given language code.
# If the model has already been loaded during this session, the cached instance is returned.
def load_vosk_model(lang: str) -> Model:
    # Returns a previously loaded model to avoid repeated disk reads and initialization cost.
    if lang in MODEL_CACHE:
        return MODEL_CACHE[lang]

    # Looks up the model directory path associated with the given language code.
    model_dir = VOSK_MODEL_PATHS[lang]

    # Ensures the expected model folder exists before attempting to initialize the model.
    if not model_dir.exists():
        raise FileNotFoundError(f"Vosk model folder not found for '{lang}': {model_dir}")

    # Initializes the Vosk Model object from the on-disk model directory.
    m = Model(str(model_dir))

    # Saves the loaded model in the cache so subsequent calls reuse the same instance.
    MODEL_CACHE[lang] = m
    return m

# Transcribes a WAV file using the Vosk model associated with the provided language code.
# Audio is standardized to the expected sample rate and int16 PCM format before recognition.
def transcribe_vosk(lang: str, wav_path: Path) -> str:
    # Retrieves the language-specific model, using the cache when available.
    model = load_vosk_model(lang)

    # Loads and preprocesses audio:
    # - converts to mono
    # - resamples to TARGET_SR
    # - normalizes loudness
    # - optionally bandpass filters for speech
    # - returns int16 PCM for recognition input
    audio_i16 = load_and_prepare_wav_i16(wav_path, TARGET_SR)

    # Creates a recognizer configured for the model and the target sample rate.
    rec = KaldiRecognizer(model, TARGET_SR)

    # Enables word-level timing/metadata in recognition results (useful for debugging and analysis).
    rec.SetWords(True)

    # Feeds the entire waveform into the recognizer as raw PCM bytes.
    # AcceptWaveform consumes the audio and prepares internal decoding state.
    rec.AcceptWaveform(audio_i16.tobytes())

    # FinalResult returns a JSON string containing the recognized text and optional word details.
    result = json.loads(rec.FinalResult())

    # Extracts the "text" field safely, ensures it is a string, and trims surrounding whitespace.
    return (result.get("text", "") or "").strip()


In [9]:
# Normalizes a transcript into a token list suitable for fair WER comparison.
# The normalization reduces differences caused by casing, punctuation, and spacing rather than speech content.
def normalize_text_for_wer(s: str) -> list:
    # Converts text to lowercase and removes leading/trailing whitespace for consistent matching.
    s = s.lower().strip()

    # Replaces curly apostrophes with straight apostrophes to avoid mismatches caused by typography.
    s = s.replace('’', "'")

    # Removes punctuation and symbols while keeping:
    # - word characters
    # - whitespace
    # - apostrophes
    # - selected accented characters common in Spanish/Italian
    # - inverted Spanish punctuation marks
    # Non-matching characters are replaced with spaces to preserve word boundaries.
    s = re.sub(r"[^\w\s'áéíóúüñàèìòùç¿¡]", " ", s, flags=re.UNICODE)

    # Collapses repeated whitespace into single spaces and trims again for stable token splitting.
    s = re.sub(r"\s+", " ", s).strip()

    # Splits into tokens on spaces; returns an empty list when the normalized string is empty.
    return s.split(" ") if s else []

# Computes Word Error Rate (WER) along with detailed counts of substitutions, deletions, and insertions.
# Uses dynamic programming (edit distance on word tokens) to find the minimum edit operations to transform ref -> hyp.
def wer_details(ref: str, hyp: str):
    # Tokenizes and normalizes the reference transcript.
    r = normalize_text_for_wer(ref)

    # Tokenizes and normalizes the hypothesis transcript produced by ASR.
    h = normalize_text_for_wer(hyp)

    # Stores token counts for DP table sizing.
    n, m = len(r), len(h)

    # dp[i][j] holds the minimum edit cost to convert the first i reference tokens into the first j hypothesis tokens.
    dp = [[0]*(m+1) for _ in range(n+1)]

    # op[i][j] stores which operation achieved the best cost at dp[i][j]:
    # E = exact match, S = substitution, I = insertion, D = deletion.
    op = [[None]*(m+1) for _ in range(n+1)]

    # Initializes the first column: converting i reference words to empty hypothesis requires i deletions.
    for i in range(1, n+1):
        dp[i][0] = i
        op[i][0] = 'D'

    # Initializes the first row: converting empty reference to j hypothesis words requires j insertions.
    for j in range(1, m+1):
        dp[0][j] = j
        op[0][j] = 'I'

    # Fills the DP tables for all prefix pairs of reference/hypothesis tokens.
    for i in range(1, n+1):
        for j in range(1, m+1):
            # If the current tokens match, no extra cost is added and the diagonal transition is chosen.
            if r[i-1] == h[j-1]:
                dp[i][j] = dp[i-1][j-1]
                op[i][j] = 'E'
            else:
                # Computes costs for the three possible edit operations:
                # substitution: replace r[i-1] with h[j-1]
                # insertion: insert h[j-1] into the reference alignment
                # deletion: delete r[i-1] from the reference alignment
                sub = dp[i-1][j-1] + 1
                ins = dp[i][j-1] + 1
                dele = dp[i-1][j] + 1

                # Selects the minimum-cost edit among substitution/insertion/deletion.
                best = min(sub, ins, dele)
                dp[i][j] = best

                # Records which operation achieved the minimum to support backtracking.
                op[i][j] = 'S' if best == sub else ('I' if best == ins else 'D')

    # Backtracks from dp[n][m] to count how many substitutions, insertions, and deletions occurred.
    i, j = n, m
    S = D = I = 0
    while i > 0 or j > 0:
        # Exact match consumes one token from both sequences without error counts.
        if i > 0 and j > 0 and op[i][j] == 'E':
            i -= 1
            j -= 1
        # Substitution consumes one token from both sequences and increments substitution count.
        elif i > 0 and j > 0 and op[i][j] == 'S':
            S += 1
            i -= 1
            j -= 1
        # Insertion consumes one hypothesis token and increments insertion count.
        elif j > 0 and op[i][j] == 'I':
            I += 1
            j -= 1
        # Deletion consumes one reference token and increments deletion count.
        else:
            D += 1
            i -= 1

    # Uses reference length as the WER denominator, protecting against divide-by-zero with a minimum of 1.
    N = max(len(r), 1)

    # Computes WER as a percentage: (S + D + I) / N * 100.
    wer = (S + D + I) / N * 100.0

    # Returns the WER percentage plus a breakdown dictionary for reporting and analysis.
    return wer, {"S": S, "D": D, "I": I, "N": N}


In [10]:
# Defines the output schema for the evaluation table so results are consistent and easy to export or report.
# Columns include the language label, file name, reference transcript, ASR hypothesis, WER percent, and edit counts.
RESULT_COLS = ["Language", "File", "Reference", "Hypothesis", "WER_%", "S", "D", "I", "N"]

# Maps internal language codes to the exact folder names used in the dataset directory structure.
# This ensures file paths are constructed correctly for each language group.
LANG_TO_FOLDER = {"en": "EN", "es": "ES", "it": "IT"}

# Evaluates speech recognition performance across all provided audio files.
# For each file:
# - selects the correct language model
# - transcribes the audio using Vosk
# - computes WER and detailed edit counts against the reference transcript
# Returns:
# - a DataFrame containing all evaluation rows
# - a list of missing files that were expected but not found on disk
def evaluate_all(audio_dir: Path):
    # Accumulates per-file result dictionaries that will become DataFrame rows.
    rows = []

    # Tracks files referenced in the mapping dictionaries but missing from the filesystem.
    missing = []

    # Iterates over every expected filename and its ground-truth reference text.
    for fname, ref in REFERENCE.items():
        # Looks up the language code for the file so the correct model and folder can be selected.
        lang = LANG_BY_FILE[fname]

        # Builds the absolute path to the WAV file based on audio root, language folder, and filename.
        wav_path = audio_dir / LANG_TO_FOLDER[lang] / fname

        # Skips evaluation when the WAV file is absent, recording it for reporting.
        if not wav_path.exists():
            missing.append(f"{LANG_TO_FOLDER[lang]}/{fname}")
            continue

        # Runs language-specific transcription using the preloaded/cached Vosk model for that language.
        hyp = transcribe_vosk(lang, wav_path)

        # Computes WER percentage and edit operation counts between reference and hypothesis at word level.
        wer, details = wer_details(ref, hyp)

        # Stores a structured row with both human-readable fields and numeric error breakdown for analysis.
        rows.append({
            "Language": {"en": "English", "it": "Italian", "es": "Spanish"}[lang],
            "File": fname,
            "Reference": ref,
            "Hypothesis": hyp,
            "WER_%": round(wer, 2),
            "S": details["S"],
            "D": details["D"],
            "I": details["I"],
            "N": details["N"],
        })

    # Builds a DataFrame using a fixed column order so downstream printing/saving stays consistent.
    df = pd.DataFrame(rows, columns=RESULT_COLS)

    # Sorts results by language then filename to produce a stable, readable table layout.
    if not df.empty:
        df = df.sort_values(["Language", "File"]).reset_index(drop=True)

    # Returns both the results table and the list of missing files for visibility.
    return df, missing

# Runs the full evaluation on the configured audio directory.
df_results, missing_files = evaluate_all(AUDIO_DIR)

# Prints the resolved absolute path to confirm which directory is being evaluated.
print("AUDIO_DIR:", AUDIO_DIR.resolve())

# Prints any missing expected files so evaluation gaps are visible immediately.
print("Missing audio files:", missing_files)

# Displays the results DataFrame as the final output of the cell.
df_results


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/EN/model-en/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/EN/model-en/graph/HCLr.fst /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/EN/model-en/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:303) Loading winfo /Users/paris/De

AUDIO_DIR: /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files
Missing audio files: []


,Language,File,Reference,Hypothesis,WER_%,S,D,I,N
0,English,checkin.wav,Where is the check-in desk?,where is the check in desk,0.00,0,0,0,6
1,English,parents.wav,I have lost my parents.,i lost my therapist,40.00,1,1,0,5
2,English,suitcase.wav,"Please, I have lost my suitcase.",please have lost my suitcase,16.67,0,1,0,6
3,English,what_time.wav,What time is my plane?,what time is my life,20.00,1,0,0,5
4,English,where.wav,Where are the restaurants and shops?,where are the restaurants and shops,0.00,0,0,0,6
5,Italian,checkin_it.wav,Dove e' il bancone?,dove è il bancone,25.00,1,0,0,4
6,Italian,parents_it.wav,Ho perso i miei genitori.,ho perso i miei genitori,0.00,0,0,0,5
7,Italian,suitcase_it.wav,"Per favore, ho perso la mia valigia.",per favore ho perso la mia valigia,0.00,0,0,0,7
8,Italian,what_time_it.wav,A che ora e’ il mio aereo?,a che ora è il mio aereo,14.29,1,0,0,7
9,Italian,where_it.wav,Dove sono i ristoranti e i negozi?,dove sono i ristoranti e i negozi,0.00,0,0,0,7


In [11]:
print("Total evaluated files:", len(df_results))
df_results


Total evaluated files: 15


,Language,File,Reference,Hypothesis,WER_%,S,D,I,N
0,English,checkin.wav,Where is the check-in desk?,where is the check in desk,0.00,0,0,0,6
1,English,parents.wav,I have lost my parents.,i lost my therapist,40.00,1,1,0,5
2,English,suitcase.wav,"Please, I have lost my suitcase.",please have lost my suitcase,16.67,0,1,0,6
3,English,what_time.wav,What time is my plane?,what time is my life,20.00,1,0,0,5
4,English,where.wav,Where are the restaurants and shops?,where are the restaurants and shops,0.00,0,0,0,6
5,Italian,checkin_it.wav,Dove e' il bancone?,dove è il bancone,25.00,1,0,0,4
6,Italian,parents_it.wav,Ho perso i miei genitori.,ho perso i miei genitori,0.00,0,0,0,5
7,Italian,suitcase_it.wav,"Per favore, ho perso la mia valigia.",per favore ho perso la mia valigia,0.00,0,0,0,7
8,Italian,what_time_it.wav,A che ora e’ il mio aereo?,a che ora è il mio aereo,14.29,1,0,0,7
9,Italian,where_it.wav,Dove sono i ristoranti e i negozi?,dove sono i ristoranti e i negozi,0.00,0,0,0,7


In [12]:
##saving the results to csv file
out_csv = Path("/Users/paris/Desktop/vosk/audio_files/Ex4_files/vosk/vosk_results.csv")

df_results.to_csv(out_csv, index=False)

print("Results saved to:", out_csv)
print("Rows saved:", len(df_results))


Results saved to: /Users/paris/Desktop/vosk/audio_files/Ex4_files/vosk/vosk_results.csv
Rows saved: 15


In [13]:
##summary of results of all 3 models
summary = (
    df_results
    .groupby("Language")["WER_%"]
    .agg(["count", "mean", "std"])
    .round(2)
)

summary


,count,mean,std
Language,,,
English,5,15.33,16.60
Italian,5,7.86,11.41
Spanish,5,26.91,20.13


In [14]:
# Defines the required custom recordings and their expected ground-truth transcripts.
# Each key is the WAV filename that should exist in the language folder, and each value is the target sentence text.
YOUR_SENTENCES = {
    "your_sentence1.wav": "I don't want anything to drink i'm rather hungry.",
    "your_sentence2.wav": "I excercise for an hour every morning.",
}

# Selects the language code that matches the recordings being evaluated.
# This value controls which language folder and which speech model are used.
YOUR_LANG = "en"          

# Looks up the dataset folder name (EN / IT / ES) corresponding to the selected language code.
YOUR_FOLDER = LANG_TO_FOLDER[YOUR_LANG]

# Prints the exact directory where the custom recordings should be placed so evaluation can find them.
print("Put your recordings in:", (AUDIO_DIR / YOUR_FOLDER))


Put your recordings in: /Users/paris/Desktop/vosk/audio_files/Ex4_files/Ex4_audio_files/EN


In [15]:
# Evaluates recognition performance on the custom user-recorded sentences defined in YOUR_SENTENCES.
# For each recording:
# - confirms the WAV file exists in the selected language folder
# - transcribes using the chosen Vosk model
# - computes WER and detailed edit counts against the provided reference text
# Returns a DataFrame aligned to the same RESULT_COLS schema used for the main evaluation.
def evaluate_your_sentences():
    # Collects per-file result rows that will be converted into a DataFrame at the end.
    rows = []

    # Iterates through each expected custom WAV filename and its ground-truth reference transcript.
    for fname, ref in YOUR_SENTENCES.items():
        # Builds the full path to the recording inside the selected language folder.
        wav_path = AUDIO_DIR / YOUR_FOLDER / fname

        # Skips evaluation if the recording is missing and prints the exact missing path for quick correction.
        if not wav_path.exists():
            print("Missing:", wav_path)
            continue

        # Runs speech recognition using the model associated with YOUR_LANG.
        hyp = transcribe_vosk(YOUR_LANG, wav_path)

        # Computes Word Error Rate percentage and a breakdown of substitutions, deletions, and insertions.
        wer, details = wer_details(ref, hyp)

        # Stores a standardized results row so this output matches the main evaluation table format.
        rows.append({
            "Language": {"en": "English", "it": "Italian", "es": "Spanish"}[YOUR_LANG],
            "File": fname,
            "Reference": ref,
            "Hypothesis": hyp,
            "WER_%": round(wer, 2),
            "S": details["S"],
            "D": details["D"],
            "I": details["I"],
            "N": details["N"],
        })

    # Builds and returns a DataFrame with the same column ordering used in other evaluation outputs.
    return pd.DataFrame(rows, columns=RESULT_COLS)

# Runs evaluation for the custom recordings and stores results in a dedicated DataFrame.
df_your = evaluate_your_sentences()

# Displays the evaluation table for the custom recordings.
df_your


,Language,File,Reference,Hypothesis,WER_%,S,D,I,N
0,English,your_sentence1.wav,I don't want anything to drink i'm rather hungry.,i don't want anything to drink i'm rather hungry,0.00,0,0,0,9
1,English,your_sentence2.wav,I excercise for an hour every morning.,i could pose for an over every morning,42.86,2,0,1,7


In [16]:
# Merges the main dataset evaluation results with the custom-recording results when available.
# If no custom results exist, the main results are used as-is.
if not df_your.empty:
    # Concatenates both DataFrames into a single table and resets the row index for a clean combined output.
    df_all = pd.concat([df_results, df_your], ignore_index=True)
else:
    # Creates a copy to avoid accidental modification of the original main results DataFrame.
    df_all = df_results.copy()

# Sorts the combined results by language and filename to produce a stable, readable ordering.
df_all = df_all.sort_values(["Language", "File"]).reset_index(drop=True)

# Writes the final combined results to a CSV file for submission, reporting, or later analysis.
df_all.to_csv(out_csv, index=False)

# Prints confirmation messages so the saved output location and row count are visible immediately.
print("FINAL CSV saved:", out_csv)
print("Total rows:", len(df_all))

# Displays the final combined results table.
df_all


FINAL CSV saved: /Users/paris/Desktop/vosk/audio_files/Ex4_files/vosk/vosk_results.csv
Total rows: 17


,Language,File,Reference,Hypothesis,WER_%,S,D,I,N
0,English,checkin.wav,Where is the check-in desk?,where is the check in desk,0.00,0,0,0,6
1,English,parents.wav,I have lost my parents.,i lost my therapist,40.00,1,1,0,5
2,English,suitcase.wav,"Please, I have lost my suitcase.",please have lost my suitcase,16.67,0,1,0,6
3,English,what_time.wav,What time is my plane?,what time is my life,20.00,1,0,0,5
4,English,where.wav,Where are the restaurants and shops?,where are the restaurants and shops,0.00,0,0,0,6
5,English,your_sentence1.wav,I don't want anything to drink i'm rather hungry.,i don't want anything to drink i'm rather hungry,0.00,0,0,0,9
6,English,your_sentence2.wav,I excercise for an hour every morning.,i could pose for an over every morning,42.86,2,0,1,7
7,Italian,checkin_it.wav,Dove e' il bancone?,dove è il bancone,25.00,1,0,0,4
8,Italian,parents_it.wav,Ho perso i miei genitori.,ho perso i miei genitori,0.00,0,0,0,5
9,Italian,suitcase_it.wav,"Per favore, ho perso la mia valigia.",per favore ho perso la mia valigia,0.00,0,0,0,7
